In [4]:
# Importando bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans


ModuleNotFoundError: No module named 'pandas'

In [5]:
# Configuração visual
sns.set(style="whitegrid")

# ==============================
# 1. Carregamento dos dados
# ==============================
customers = pd.read_csv('olist_customers_dataset.csv')
orders = pd.read_csv('olist_orders_dataset.csv')
order_items = pd.read_csv('olist_order_items_dataset.csv')
order_payments = pd.read_csv('olist_order_payments_dataset.csv')
# ==============================


NameError: name 'sns' is not defined

In [6]:

# 2. Pré-processamento
# ==============================
# Filtrar pedidos entregues
orders = orders[orders['order_status'] == 'delivered'].copy()

# Converter datas
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
# ==============================

NameError: name 'orders' is not defined

In [7]:


# 3. Agregações
# ==============================
payments_per_order = (
order_payments.groupby('order_id')['payment_value']
.sum()
.reset_index()
)

items_per_order = (
order_items.groupby('order_id')['price']
.sum()
.reset_index()
)

# Merge das bases
orders = orders.merge(payments_per_order, on='order_id', how='left')
orders = orders.merge(items_per_order, on='order_id', how='left')
orders = orders.merge(customers, on='customer_id', how='left')

# ==============================

NameError: name 'order_payments' is not defined

In [8]:
# 4. Base por cliente
# ==============================
df_clientes = orders.groupby('customer_id').agg({
'order_id': 'count',
'payment_value': 'sum', # MAIS CORRETO
'order_purchase_timestamp': 'max',
'customer_city': 'first',
'customer_state': 'first'
}).reset_index()

df_clientes.rename(columns={
'order_id': 'total_pedidos',
'payment_value': 'valor_total',
'order_purchase_timestamp': 'ultima_compra'
}, inplace=True)

# Ticket médio
df_clientes['ticket_medio'] = df_clientes['valor_total'] / df_clientes['total_pedidos']

# ==============================

NameError: name 'orders' is not defined

In [18]:
# 5. Análise exploratória
# ==============================

# Top estados
clientes_estado = df_clientes['customer_state'].value_counts().head(10)

plt.figure(figsize=(10,5))
sns.barplot(x=clientes_estado.index, y=clientes_estado.values)
plt.title("Distribuição de Clientes por Estado")
plt.xlabel("Estado")
plt.ylabel("Quantidade")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Tipo de cliente
df_clientes['tipo_cliente'] = np.where(
    df_clientes['total_pedidos'] > 1, 'Recorrente', 'Único'
)

plt.figure(figsize=(6,4))
df_clientes['tipo_cliente'].value_counts(normalize=True).plot(kind='bar')
plt.title("Frequência de Compra")
plt.ylabel("Proporção")
plt.xticks(rotation=0)
plt.show()

# Garantir ticket médio
df_clientes['ticket_medio'] = df_clientes['valor_total'] / df_clientes['total_pedidos']

plt.figure(figsize=(6,4))
sns.boxplot(x='tipo_cliente', y='ticket_medio', data=df_clientes)
plt.title("Ticket Médio por Tipo de Cliente")
plt.xlabel("Tipo de Cliente")
plt.ylabel("Ticket Médio")
plt.show()

# ==============================

NameError: name 'df_clientes' is not defined

In [ ]:
# 6. RFM
# ==============================
import pandas as pd
import matplotlib.pyplot as plt

# Garantir que a coluna é datetime
df_clientes['ultima_compra'] = pd.to_datetime(df_clientes['ultima_compra'])

# Data de referência
snapshot_date = df_clientes['ultima_compra'].max() + pd.Timedelta(days=1)

# Métricas RFM
df_clientes['recencia'] = (snapshot_date - df_clientes['ultima_compra']).dt.days
df_clientes['frequencia'] = df_clientes['total_pedidos']
df_clientes['monetario'] = df_clientes['valor_total']

# Função segura para qcut
def calcular_score(coluna, labels):
    return pd.qcut(
        coluna,
        q=5,
        labels=labels,
        duplicates='drop'
    ).astype(float).fillna(0).astype(int)

# Scores RFM
df_clientes['R_score'] = calcular_score(df_clientes['recencia'], range(5, 0, -1))
df_clientes['F_score'] = calcular_score(df_clientes['frequencia'], range(1, 6))
df_clientes['M_score'] = calcular_score(df_clientes['monetario'], range(1, 6))

# ==============================
# Segmentação RFM
# ==============================

def segment_rfm(row):
    if row['R_score'] >= 4 and row['F_score'] >= 4 and row['M_score'] >= 4:
        return 'Campeão'
    elif row['F_score'] >= 4:
        return 'Fiel'
    elif row['R_score'] <= 2:
        return 'Em Risco'
    else:
        return 'Outros'

df_clientes['segmento'] = df_clientes.apply(segment_rfm, axis=1)

# ==============================
# Visualização
# ==============================

plt.figure(figsize=(8, 5))
df_clientes['segmento'].value_counts().plot(kind='bar')
plt.title("Segmentos RFM")
plt.xlabel("Segmento")
plt.ylabel("Quantidade")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'pandas'

In [17]:
# 7. Clusterização (K-Means)
# ==============================

rfm_data = df_clientes[['recencia','frequencia','monetario']]

# Garantir valores positivos
rfm_data = rfm_data.clip(lower=0)

# Log
rfm_log = np.log1p(rfm_data)

# Padronização
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)

# Modelo
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df_clientes['cluster'] = kmeans.fit_predict(rfm_scaled)

# Visualização
plt.figure(figsize=(8,5))
sns.scatterplot(
    x='recencia',
    y='monetario',
    hue='cluster',
    data=df_clientes,
    palette='Set2'
)
plt.title("Clusterização de Clientes (K-Means)")
plt.show()
# ==============================

NameError: name 'df_clientes' is not defined

In [11]:
# 8. Análise dos clusters
# ==============================
cluster_summary = df_clientes.groupby('cluster')[[
'recencia','frequencia','monetario'
]].mean()

print(cluster_summary)

NameError: name 'df_clientes' is not defined

In [16]:
# estrategias_negocio.py

class EstrategiasNegocio:
    def __init__(self):
        self.estrategias = {
            "Retenção de Clientes": [
                "Programas de fidelidade",
                "Incentivos para segunda compra",
                "Comunicação personalizada"
            ],
            "Reativação de Clientes": [
                "Campanhas de e-mail marketing",
                "Cupons de desconto",
                "Remarketing"
            ],
            "Maximização de Receita": [
                "Programas VIP",
                "Ofertas exclusivas",
                "Upsell",
                "Cross-sell"
            ],
            "Expansão Geográfica": [
                "Investimento logístico",
                "Parcerias regionais",
                "Campanhas direcionadas"
            ]
        }

    def exibir_estrategias(self):
        print("\n📊 Estratégias de Crescimento\n")
        for categoria, itens in self.estrategias.items():
            print(f"🔹 {categoria}:")
            for item in itens:
                print(f" - {item}")
            print()


if __name__ == "__main__":
    negocio = EstrategiasNegocio()
    negocio.exibir_estrategias()


📊 Estratégias de Crescimento

🔹 Retenção de Clientes:
 - Programas de fidelidade
 - Incentivos para segunda compra
 - Comunicação personalizada

🔹 Reativação de Clientes:
 - Campanhas de e-mail marketing
 - Cupons de desconto
 - Remarketing

🔹 Maximização de Receita:
 - Programas VIP
 - Ofertas exclusivas
 - Upsell
 - Cross-sell

🔹 Expansão Geográfica:
 - Investimento logístico
 - Parcerias regionais
 - Campanhas direcionadas

